# Functions — Polyglot Reference

How you declare, call, pass, and capture functions. Parameters, lambdas, function types, closures, and the higher-order patterns built on top.

**Languages:** Java · Scala · Kotlin · JavaScript · TypeScript · Python

This notebook covers:

1. **Declaration** — keyword, return-type position, top-level vs class-bound, expression body
2. **Parameters** — defaults, named arguments, varargs, keyword-collecting params
3. **Function values & lambdas** — first-class function syntax, function types, method references
4. **Closures & higher-order patterns** — variable capture, mutability of captured state, currying, composition
5. **Tail calls, async, generators** — recursion guarantees, suspendable functions, lazy iteration

Concurrency mental models — JavaScript's event loop, Kotlin coroutines, Python's asyncio scheduler, Scala `Future` execution contexts, Java `CompletableFuture` — are deliberately out of scope. They are too divergent to compare side-by-side honestly. Per-language repos cover each. Pattern matching with destructured function parameters is in `08-classes-inheritance-matching.ipynb`. Statement-vs-expression positioning of function bodies is in `03-operators-expressions.ipynb`.

## Declaration

Where a function can live and what its signature looks like. Java is the loudest outlier — every callable must be a method on a class or interface. The other five all allow free top-level functions.

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| keyword | *(none — type before name)* | `def` | `fun` | `function` *(or arrow)* | same | `def` |
| top-level function | — *(static method only)* | yes | yes | yes | yes | yes |
| return-type position | before name: `int f(...)` | after `:` — `def f(...): Int` | after `:` — `fun f(...): Int` | — *(untyped)* | after `:` — `function f(...): number` | after `->` — `def f(...) -> int:` *(hint only)* |
| no-return type | `void` | `Unit` | `Unit` | (returns `undefined`) | `void` | (returns `None`) |
| expression body | — | `def f(x: Int) = x * 2` | `fun f(x: Int) = x * 2` | `(x) => x * 2` | same | `lambda x: x * 2` *(expr only)* |
| explicit return | `return x;` *(required)* | `return x` *(rare; last expr wins)* | `return x` *(or expr body)* | `return x;` | same | `return x` |
| forward reference | — *(class members visible in any order)* | scope-aware | scope-aware | hoisted *(`function` decl)* | same | not hoisted |
| overloading by signature | yes | yes | yes | — | declaration overloads only | — |
| multiple return values | — *(class or `record`)* | tuple `(a, b)` | `Pair` / `Triple` / `data class` | array / object destructure | tuple type | tuple `return a, b` |

Java's *everything-is-a-method* is the loudest divergence — there is no top-level `void main` until Java 21's instance-main proposal, and even then the JVM still hosts it inside a class. JavaScript's hoisting rules are the next-loudest: a `function f() {}` declaration is fully hoisted, an arrow-function-assigned-to-`const` is not.

## Parameters

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| default value | — | `x: Int = 0` | `x: Int = 0` | `x = 0` | same | `x=0` |
| named argument at call site | — | `f(x = 1)` | `f(x = 1)` | — *(use object literal)* | — | `f(x=1)` |
| varargs | `T... xs` | `xs: T*` | `vararg xs: T` | `...xs` | same | `*args` |
| keyword-collecting param | — | — | — | object-destructure workaround | same | `**kwargs` |
| destructured parameter | — | pattern in `def` body | destructured `data class` | `({a, b}) => ...` | same | `def f(*, a, b)` *(keyword-only)* |
| trailing comma in signature | yes *(8+)* | yes | yes | yes | yes | yes |
| positional-only marker | — | — | — | — | — | `def f(a, /, b)` *(3.8+)* |
| `this` parameter | implicit | implicit | implicit | implicit *(rebindable)* | `this: T` annotation | explicit `self` |

Java is the only one without default parameters or named arguments — the workaround is method overloading or the builder pattern. JavaScript's lack of named arguments is patched culturally with single-object-literal-as-arg, which TypeScript types as a destructured parameter object. Python's explicit `self` is unique among these six and is a deliberate readability choice from PEP 8 — no other language here makes the receiver visible in the signature.

## Function Values & Lambdas

A *function value* is a callable that you can store in a variable, pass as an argument, or return from another function. A *lambda* is the syntax for writing one inline. Java did not have either until 8 — and even now, Java's *function values* are objects implementing single-method *functional interfaces*, not true first-class functions.

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| lambda syntax | `(x) -> x + 1` | `(x: Int) => x + 1` | `{ x: Int -> x + 1 }` | `(x) => x + 1` | same | `lambda x: x + 1` |
| multi-statement body | `(x) -> { ...; return ...; }` | `(x) => { ...; lastExpr }` | `{ x -> ...; lastExpr }` | `(x) => { ...; return ...; }` | same | — *(use named `def`)* |
| function type | `Function<Integer,String>` | `Int => String` | `(Int) -> String` | — *(structural)* | `(x: number) => string` | `Callable[[int], str]` |
| zero-arg lambda | `() -> e` | `() => e` | `{ -> e }` *(or `{ e }`)* | `() => e` | same | `lambda: e` |
| implicit single param | — | `_` placeholder | `it` | — | — | — |
| method reference | `String::length` | `_.length` *(eta-expansion)* | `String::length` | `obj.method` *(rebinds `this`)* | same | `obj.method` |
| trailing-lambda call | — | — | `xs.map { it * 2 }` | — | — | — |
| inline / no-allocation | — *(JIT may)* | — *(JIT may)* | `inline fun` *(source-level)* | — | — | — |

Kotlin's trailing-lambda rule is the most idiom-shifting line on this table: if the last argument is a lambda, you write it after the parens — `xs.fold(0) { a, b -> a + b }`. Combined with `it` as the implicit single-parameter name, it is what makes Kotlin code read so close to Ruby or Swift.

## Closures & Higher-Order Patterns

Every language here has closures — functions that capture variables from their enclosing scope. The interesting axis is *what you can do with the captured variable*.

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| read captured var | yes | yes | yes | yes | yes | yes |
| capture must be final | yes — *effectively final* | no | no | no | no | no |
| mutate captured var | — *(wrap in array / `AtomicReference`)* | yes — capture a `var` | yes — capture a `var` | `let` / `var` | same | `nonlocal x` |
| nested function | — *(local class workaround)* | yes | yes | yes | yes | yes |
| name resolution | lexical | lexical | lexical | lexical | lexical | LEGB *(lexical with rules)* |

Java requires captured locals to be *effectively final* — once assigned, never reassigned. The language designers refused to make captured mutability easy because closures sharing mutable state across thread boundaries is one of the most common JVM concurrency bugs. Python's `nonlocal x` and `global x` declarations are explicit on purpose for the same reason — assignment inside a function defaults to creating a *local* variable, and you must opt in to writing back to the enclosing scope.

The most common cross-language footgun is **late binding in loops**. Python and JavaScript-with-`var` famously share this: a closure created inside a `for` loop captures the *variable*, not its value at creation time. By the time the closure runs, the loop has finished and the variable holds its final value.

```
fns = [lambda: i for i in range(3)]
[f() for f in fns]   # [2, 2, 2]  — not [0, 1, 2]
```

Fix: bind via default argument — `lambda i=i: i` — in Python, or use `let` instead of `var` in JavaScript. Scala, Kotlin, and Java avoid this because their for-each loop variable is a fresh binding per iteration.

### Higher-order patterns

| Pattern | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| currying *(declaration syntax)* | — | `def f(a: Int)(b: Int)` | — | — | — | — |
| partial application | manual lambda | `f(_: Int, 5)` | manual lambda | `f.bind(null, 5)` | same | `functools.partial(f, 5)` |
| function composition | `f.andThen(g)` | `f andThen g` / `g compose f` | `g compose f` *(stdlib)* | manual `(x) => g(f(x))` | same | manual or 3rd-party |
| memoization | manual cache map | manual / library | manual | manual | same | `functools.lru_cache` |
| receiver / extension function | — | `implicit class` | `fun T.f()` | — | same *(via `this`)* | — |

Scala is the only one with currying baked into the *declaration* syntax. Everywhere else, you write a function that returns a function. Kotlin's *extension function* — `fun String.lastWord(): String` — adds a method-call-syntax function to a type without modifying it. Useful when reaching for currying or method-chain ergonomics in other languages.

## Tail Calls, Async, Generators

Three orthogonal extensions to the basic function model: stack-safe recursion, suspendable execution, lazy iteration.

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| tail-call optimization | — | `@tailrec` *(self-calls only)* | `tailrec fun` *(self-calls only)* | spec yes / engines no | same | — |
| stack overflow on deep recursion | yes | unless `@tailrec` | unless `tailrec` | yes | yes | yes *(default ~1000)* |
| async function declaration | — *(`CompletableFuture`)* | — *(`Future`)* | `suspend fun` | `async function` | same | `async def` |
| await keyword | — | `Await.result` *(blocking)* | implicit at suspension point | `await` | same | `await` |
| generator function | — *(use `Stream.generate`)* | `LazyList` | `sequence { yield(x) }` | `function*` with `yield` | same | `def` with `yield` |
| yield single / many | — | `LazyList.cons(...)` | `yield(x)` / `yieldAll(xs)` | `yield x` / `yield* xs` | same | `yield x` / `yield from xs` |

Concurrency mental models below the keyword level — event loop, coroutine scheduler, `Future` execution context — are deliberately out of scope. The surface syntax converges on `async`/`await` in three of these languages; the runtime model underneath is wildly different. Per-language repos cover each.

## Notes — when a cell isn't enough

**Java's *functional interfaces* are not first-class functions.** A Java lambda compiles to an instance of a single-abstract-method interface — `Function<T, R>`, `Predicate<T>`, `Consumer<T>`, `Supplier<T>`, `BiFunction<T, U, R>`, and friends. Each parameter-arity / return-shape combination has its own interface. Calling one is `f.apply(x)`, not `f(x)`. The *useful* thing is that any interface with one abstract method can serve as a lambda target — so `Runnable`, `Callable`, `Comparator`, and your own `@FunctionalInterface` types all compose. The *annoying* thing is that primitive types break the symmetry: `IntFunction<R>`, `ToIntFunction<T>`, `IntUnaryOperator`, etc. exist to avoid boxing on the hot path.

**JavaScript's `this` binding rules.** In a `function() {}`, `this` is determined by the *call site* — `obj.method()` binds `this` to `obj`, but `const m = obj.method; m()` loses the binding (`this` becomes `undefined` in strict mode). Arrow functions are different — they do not have their own `this`; they capture the surrounding scope's `this` lexically, like every other language. The rule of thumb: write `=>` for callbacks and inline functions; write `function` only when you need a method on an object literal or class. Method shorthand in object literals (`{ method() { ... } }`) and class methods are also `function`-style.

**Kotlin `inline`, `crossinline`, `noinline`.** Marking `fun foo(action: () -> Unit)` as `inline` causes the compiler to substitute the lambda body at every call site, eliminating the lambda allocation and the call overhead. This is what makes Kotlin's collection ops (`map`, `filter`, `forEach`) competitive with hand-written loops. `crossinline` forbids non-local return from the lambda parameter; `noinline` opts a single parameter out of inlining. You will see `inline` constantly in the standard library — read it as *this lambda call has zero overhead*.

**Scala methods vs functions vs eta-expansion.** A Scala `def` defines a *method*, not a function value. Methods become function values via *eta-expansion* — `xs.map(f _)` in Scala 2, automatic in most positions in Scala 3. `def f(x: Int) = x + 1` is a method; `val g: Int => Int = f` eta-expands it into a `Function1[Int, Int]`. Most of the time you do not notice; in higher-order code with overloading or implicit conversions, the difference matters.

**Python `lambda` is expression-only.** `lambda x: x + 1` is a single-expression function. There is no multi-line `lambda` — for that, use `def`. The PEP rationale is that Python explicitly does not want anonymous multi-statement function values; if it has a body, it should have a name. The expression-only restriction is also why `lambda` is rare in idiomatic Python — `def` is short enough that there is little reason to prefer `lambda` for non-trivial functions.

**Default mutable arguments — Python footgun.** `def f(x=[]):` evaluates the `[]` *once at definition time* and reuses the same list across all calls. Idiomatic fix: `def f(x=None): x = x if x is not None else []`. Most static analysis tools flag this. None of the other five languages here share this footgun — Scala, Kotlin, JavaScript, TypeScript all evaluate default arguments per call.

**JavaScript hoisting and the temporal dead zone.** A `function foo() {}` declaration is hoisted to the top of its scope — you can call it before its textual position in the source. A `const foo = () => {}` or `const foo = function() {}` is not — calling it before the assignment line throws `ReferenceError` (the *temporal dead zone*). This is why callbacks defined with `function` work even when written below their use; arrow-assigned-to-`const` callbacks must be defined first.

**Closure over loop variable — `var` vs `let`.** Pre-ES6 JavaScript `for (var i = 0; ...)` shares one binding across iterations — closures all capture the same `i` and see the final value. ES6 `let` makes the binding fresh per iteration. This is the same footgun as Python's late binding in comprehensions; Python's fix is the default-argument trick `lambda i=i: i`.

**Tail-call optimization is rare on the JVM and absent from JavaScript engines.** Scala's `@tailrec` annotation and Kotlin's `tailrec` modifier guarantee TCO — but only for *direct self-recursion*. Mutual recursion is not optimized. The JavaScript spec mandates TCO since ES6, but no major engine implements it. Python and Java do not optimize tail calls at all. The portable answer for deep recursion is to convert to a loop or use an explicit stack.

**`async` as syntactic sugar over very different runtimes.** JavaScript's `async function` returns a `Promise`; an `await` expression suspends until the promise settles. Python's `async def` returns a coroutine; `await` suspends until the awaited coroutine yields. Kotlin's `suspend fun` requires a coroutine context and the suspension is invisible at the call site. Scala uses `Future` with `Await.result` *(blocking, discouraged)* or `for`-comprehensions over futures. The surface syntax converges; the underlying scheduling, cancellation, and exception-propagation models are wildly different. Cover that in per-language repos.

**Generator functions and lazy iteration.** A generator function suspends at each `yield` and resumes on next iteration. Python's `def` with `yield` is the most natural form. JavaScript's `function*` with `yield` is the same idea, returning an iterator. Kotlin builds it inside a `sequence { yield(x) }` block. Scala uses lazy collections (`LazyList`, formerly `Stream`) directly — there is no `yield` keyword. Java has no generator syntax — use `Stream.generate`, `Stream.iterate`, or build a `Spliterator` directly.